# 118 — Mirascope: Pydantic-First LLM Calls with Provider Switching
## What you'll learn: typed prompts, decorated LLM calls, and one-line provider swaps
⏱ ~40 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/118-mirascope-typed-llm/mirascope_typed_llm_workbook.ipynb)

Mirascope is a Pydantic-first LLM toolkit where decorators do the heavy lifting:
- `@openai.call(model="gpt-4o-mini", response_model=MyModel)` — typed LLM call
- `@prompt_template("Summarize: {text}")` — type-safe prompt with parameter validation
- Swap `@openai.call` for `@anthropic.call` — same function, different provider

This workshop demonstrates all three patterns by extracting a `MeetingSummary` from a transcript using both OpenAI and Anthropic, then comparing results.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — Mirascope vs instructor, design philosophy |
| 2 | **Setup** — install, API keys |
| 3 | **@prompt_template** — typed prompt construction |
| 4 | **@openai.call** — typed LLM invocation |
| 5 | **response_model** — Pydantic output parsing |
| 6 | **Provider switching** — @anthropic.call, same function |
| 7 | **Side-by-side comparison** — OpenAI vs Anthropic outputs |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` in `.env` or Colab Secrets
- `mirascope[openai,anthropic]`

### Key References
> [Mirascope docs](https://mirascope.io/docs)
>
> [Mirascope GitHub](https://github.com/Mirascope/mirascope)
>
> [Pydantic BaseModel](https://docs.pydantic.dev/latest/concepts/models/)

## Part 1 — Concepts: Mirascope vs Instructor

Both Mirascope and instructor extract typed outputs from LLMs. They differ in philosophy:

### Instructor: wrap the client

```python
client = instructor.from_openai(OpenAI())
result = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=MyModel,
    messages=[{"role": "user", "content": prompt}]
)
```

- Thin wrapper on the OpenAI client
- Prompts are plain strings/message dicts
- Retry logic built in
- Provider switch: swap the client factory

### Mirascope: decorate the function

```python
@openai.call(model="gpt-4o-mini", response_model=MyModel)
@prompt_template("Summarize: {transcript}")
def summarize(transcript: str): ...

result = summarize("Q3 meeting transcript...")
```

- Decorator-first: the function IS the LLM call
- Prompts are typed templates with parameter validation
- Provider switch: swap the outer decorator
- More opinionated, Pydantic-first from the ground up

### When to choose Mirascope

- You want the function signature to be the complete API contract
- You prefer decorator-based configuration over client-based
- You're building reusable typed LLM functions across a codebase
- Provider portability is a first-class concern

### When to choose instructor

- You already have OpenAI client code and want Pydantic validation
- You need fine-grained retry logic
- You want minimal code change to existing LLM calls

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "mirascope[openai,anthropic]==1.13.3",
         "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local — skipping install (using requirements.txt)")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

openai_key = os.environ.get("OPENAI_API_KEY", "")
anthropic_key = os.environ.get("ANTHROPIC_API_KEY", "")
print(f"OpenAI key ready:    {bool(openai_key) and openai_key.startswith('sk-')}")
print(f"Anthropic key ready: {bool(anthropic_key) and anthropic_key.startswith('sk-ant')}")

## Part 3 — Defining the Output Model and Transcript

The `MeetingSummary` model defines what we want to extract. Each `Field(description=...)` is shown to the LLM as a hint about what to fill in.

In [ ]:
from pydantic import BaseModel, Field


class MeetingSummary(BaseModel):
    topic: str = Field(description="The main topic or purpose of the meeting")
    key_points: list[str] = Field(
        description="The most important points discussed, as concise bullet items"
    )
    next_steps: list[str] = Field(
        description="Concrete next steps or action items agreed upon"
    )
    mood: str = Field(
        description="Overall tone of the meeting: productive, tense, inconclusive, etc."
    )


SAMPLE_TRANSCRIPT = """
Product roadmap sync — June 18, 2024

Attendees: Alex (CPO), Jamie (Engineering), Sam (Design), Riley (Marketing)

Alex noted Q2 launch hit 94% of target metrics — strong result.
The team discussed H2 roadmap. Jamie flagged real-time collaboration requires a backend
rewrite: estimated 8 weeks. Sam presented three UI directions; team voted unanimously for
option B (minimal dark-mode design). Riley raised concerns about October launch conflicting
with a competitor announcement — suggested September 15th. Alex agreed.
Action items: Jamie revised estimate by Friday; Sam design specs by June 25th;
Riley press release outline by June 30th.
"""

print("MeetingSummary fields:")
for name, field in MeetingSummary.model_fields.items():
    desc = field.description or "no description"
    print(f"  {name}: {desc[:60]}")

## Part 4 — @openai.call with response_model

### Decorator stack (read bottom to top)

```
@openai.call(model="gpt-4o-mini", response_model=MeetingSummary)   # step 2
@prompt_template("... {transcript} ...")                             # step 1
def summarize_with_openai(transcript: str):
    return {"computed_fields": {"transcript": transcript}}
```

1. Function body returns `{"computed_fields": {...}}` — values injected into the template
2. `@prompt_template` formats the template string with those values
3. `@openai.call` sends the formatted messages to OpenAI, parses into `MeetingSummary`

The return type of calling `summarize_with_openai(text)` is `MeetingSummary` — not a string, not a dict.

In [ ]:
from mirascope.core import openai, prompt_template


@openai.call(model="gpt-4o-mini", response_model=MeetingSummary)
@prompt_template(
    """
    You are an expert meeting analyst.
    Summarize the following meeting transcript into structured output.

    Transcript:
    {transcript}
    """
)
def summarize_with_openai(transcript: str) -> openai.OpenAIDynamicConfig:
    return {"computed_fields": {"transcript": transcript}}


print("Calling summarize_with_openai...")
openai_result = summarize_with_openai(SAMPLE_TRANSCRIPT)

print(f"Return type: {type(openai_result).__name__}")
print(f"\nTopic:      {openai_result.topic}")
print(f"Mood:       {openai_result.mood}")
print(f"Key points ({len(openai_result.key_points)}):")
for p in openai_result.key_points:
    print(f"  - {p}")
print(f"Next steps ({len(openai_result.next_steps)}):")
for s in openai_result.next_steps:
    print(f"  - {s}")

## Part 5 — Provider Switching: @anthropic.call

The ONLY change to switch from OpenAI to Anthropic is swapping the outer decorator.
Everything else — `@prompt_template`, function body, `response_model`, return type — is identical.

In [ ]:
from mirascope.core import anthropic


@anthropic.call(model="claude-haiku-4-5-20251001", response_model=MeetingSummary)
@prompt_template(
    """
    You are an expert meeting analyst.
    Summarize the following meeting transcript into structured output.

    Transcript:
    {transcript}
    """
)
def summarize_with_anthropic(transcript: str) -> anthropic.AnthropicDynamicConfig:
    return {"computed_fields": {"transcript": transcript}}


if not anthropic_key:
    print("ANTHROPIC_API_KEY not set — skipping Anthropic call.")
    anthropic_result = None
else:
    print("Calling summarize_with_anthropic (same prompt_template, different decorator)...")
    anthropic_result = summarize_with_anthropic(SAMPLE_TRANSCRIPT)

    print(f"Return type: {type(anthropic_result).__name__}")
    print(f"\nTopic:      {anthropic_result.topic}")
    print(f"Mood:       {anthropic_result.mood}")
    print(f"Key points ({len(anthropic_result.key_points)}):")
    for p in anthropic_result.key_points:
        print(f"  - {p}")
    print(f"Next steps ({len(anthropic_result.next_steps)}):")
    for s in anthropic_result.next_steps:
        print(f"  - {s}")

## Part 6 — Side-by-Side Comparison

Both models receive identical prompts and produce the same `MeetingSummary` type.
The comparison reveals differences in verbosity, granularity, and phrasing — not structure.

In [ ]:
def compare_summaries(oai: MeetingSummary, ant: MeetingSummary) -> None:
    W = 38
    print(f"\n{'Field':<20} {'OpenAI (gpt-4o-mini)':<{W}} {'Anthropic (haiku)'}")
    print("-" * (20 + W + 40))
    print(f"{'topic':<20} {oai.topic[:W-1]:<{W}} {ant.topic[:W-1]}")
    print(f"{'mood':<20} {oai.mood:<{W}} {ant.mood}")

    for i in range(max(len(oai.key_points), len(ant.key_points))):
        o = oai.key_points[i][:W-1] if i < len(oai.key_points) else "(none)"
        a = ant.key_points[i][:W-1] if i < len(ant.key_points) else "(none)"
        print(f"  {'key_points['+str(i)+']':<18} {o:<{W}} {a}")

    for i in range(max(len(oai.next_steps), len(ant.next_steps))):
        o = oai.next_steps[i][:W-1] if i < len(oai.next_steps) else "(none)"
        a = ant.next_steps[i][:W-1] if i < len(ant.next_steps) else "(none)"
        print(f"  {'next_steps['+str(i)+']':<18} {o:<{W}} {a}")


if anthropic_result is not None:
    compare_summaries(openai_result, anthropic_result)
else:
    print("Anthropic result not available.")
    print("OpenAI result:")
    print(f"  topic:      {openai_result.topic}")
    print(f"  mood:       {openai_result.mood}")
    print(f"  key_points: {openai_result.key_points}")
    print(f"  next_steps: {openai_result.next_steps}")

## Part 7 — Streaming

Mirascope also supports streaming by adding `stream=True` to the call decorator.
With streaming, the function returns an iterator of `(chunk, tool)` tuples.
This is useful for real-time UIs where you want to show partial output as it arrives.

In [ ]:
@openai.call(model="gpt-4o-mini", stream=True)
@prompt_template("Write a 2-sentence summary of the meeting topic: {topic}")
def stream_summary(topic: str) -> openai.OpenAIDynamicConfig:
    return {"computed_fields": {"topic": topic}}


print("Streaming response (gpt-4o-mini):")
print("-" * 50)
for chunk, _ in stream_summary("product roadmap planning with Q2 review and H2 goal setting"):
    print(chunk.content, end="", flush=True)
print("\n" + "-" * 50)

## Exercises

### Exercise 1 — New output model

Define an `ActionItems` Pydantic model with:
- `items: list[str]` — concrete action items
- `owners: list[str]` — person responsible for each item
- `urgent_count: int` — number of items with a deadline within 2 weeks of the meeting

Create an `@openai.call` decorated function that extracts `ActionItems` from the same transcript.

---

### Exercise 2 — Two-variable prompt template

Create a prompt template with TWO variables: `transcript` and `max_points: int`.
Template: `"Extract at most {max_points} key points from: {transcript}"`
Call it with `max_points=2`. Does the model respect the constraint?

---

### Exercise 3 — Mirascope vs instructor comparison

Implement the same `MeetingSummary` extraction using instructor (example 117 pattern):
```python
client = instructor.from_openai(OpenAI())
result = client.chat.completions.create(
    model="gpt-4o-mini",
    response_model=MeetingSummary,
    messages=[...]
)
```
Count the lines of code for each approach. What does each approach hide/expose?

In [ ]:
# ===== ANSWER KEY — Exercise 1: ActionItems model =====

class ActionItems(BaseModel):
    items: list[str] = Field(description="Concrete action items from the meeting")
    owners: list[str] = Field(
        description="Person responsible for each action item (same order as items)"
    )
    urgent_count: int = Field(
        description="Number of action items with a deadline within 2 weeks of June 18, 2024"
    )


@openai.call(model="gpt-4o-mini", response_model=ActionItems)
@prompt_template(
    """
    Extract action items, their owners, and count urgent items (deadline within 2 weeks
    of June 18, 2024) from this meeting transcript:

    {transcript}
    """
)
def extract_action_items(transcript: str) -> openai.OpenAIDynamicConfig:
    return {"computed_fields": {"transcript": transcript}}


ai = extract_action_items(SAMPLE_TRANSCRIPT)
print("ActionItems extracted:")
print(f"  urgent_count: {ai.urgent_count}")
for i, (item, owner) in enumerate(zip(ai.items, ai.owners)):
    print(f"  [{owner}] {item}")

In [ ]:
# ===== ANSWER KEY — Exercise 2: Two-variable prompt template =====

@openai.call(model="gpt-4o-mini")
@prompt_template(
    """
    Extract at most {max_points} key points from this meeting transcript.
    Return ONLY a numbered list of exactly {max_points} points or fewer.

    Transcript:
    {transcript}
    """
)
def extract_limited_points(transcript: str, max_points: int) -> openai.OpenAIDynamicConfig:
    return {"computed_fields": {"transcript": transcript, "max_points": max_points}}


result_2pts = extract_limited_points(SAMPLE_TRANSCRIPT, max_points=2)
print("Limited extraction (max_points=2):")
print(result_2pts.content)

result_4pts = extract_limited_points(SAMPLE_TRANSCRIPT, max_points=4)
print("\nLimited extraction (max_points=4):")
print(result_4pts.content)

In [ ]:
# ===== ANSWER KEY — Exercise 3: Mirascope vs instructor comparison =====

import instructor
from openai import OpenAI

# ---- instructor version ----
instr_client = instructor.from_openai(OpenAI())

def extract_with_instructor(transcript: str) -> MeetingSummary:
    return instr_client.chat.completions.create(
        model="gpt-4o-mini",
        response_model=MeetingSummary,
        max_retries=3,
        messages=[
            {"role": "system", "content": "You are an expert meeting analyst."},
            {"role": "user", "content": f"Summarize this meeting:\n\n{transcript}"},
        ],
    )

instr_result = extract_with_instructor(SAMPLE_TRANSCRIPT)

print("instructor result:")
print(f"  topic: {instr_result.topic}")
print(f"  mood:  {instr_result.mood}")

print()
print("Code comparison (lines of setup code):")
print("  Mirascope: ~8 lines (2 decorators + function body)")
print("  instructor: ~10 lines (client creation + create() call)")
print()
print("Key differences:")
print("  Mirascope: function IS the LLM call; prompt is part of the definition")
print("  instructor: prompt is passed at call-time; client is configured separately")
print("  Mirascope: provider swap = 1 decorator change")
print("  instructor: provider swap = client factory change (from_openai -> from_anthropic)")

## Workshop Complete

You have mastered Mirascope's core patterns:

- **`@prompt_template`** — typed prompt construction with parameter validation
- **`@openai.call(response_model=MyModel)`** — the decorator stack that turns a function into a typed LLM call
- **`OpenAIDynamicConfig`** — return type for injecting computed values into templates
- **Provider switching** — swap `@openai.call` for `@anthropic.call` with no other changes
- **Streaming** — `stream=True` returns an iterator of chunks

**Key insight**: Mirascope and instructor solve the same problem (typed LLM outputs) with different philosophies. Mirascope is more opinionated and decorator-heavy; instructor is thinner and wraps existing clients. Both produce validated Pydantic objects and support multiple providers.

---

Next: continue with the AI frameworks series — comparison of orchestration approaches.

---
### Further reading
- [Mirascope docs](https://mirascope.io/docs)
- [Mirascope GitHub](https://github.com/Mirascope/mirascope)
- [Mirascope vs instructor comparison](https://mirascope.io/docs/compare/instructor/)
- [Pydantic structured output patterns](https://docs.pydantic.dev/latest/)</cell id="cell-md-020"></cell>
